* This notebook implements an MLP from scratch and then with concise PyTorch modules, keeping the mapping between the 2 visible.

In [ ]:
import math
import random
import numpy as np
import torch

# 5.2.1 Implementation from Scratch

## 1. Intuition

*   From scratch, an MLP stores weight matrices and bias vectors for each layer and defines the forward computation manually.

*   For classification, the final layer outputs one logit per class.

## 2. Why this exists

*   Manual implementation exposes what framework layers store and compute internally.

## 3. Examples

*   Define parameters and a forward function for a tiny MLP.


In [ ]:
W1 = torch.randn(4, 5) * 0.01
b1 = torch.zeros(5)
W2 = torch.randn(5, 3) * 0.01
b2 = torch.zeros(3)

def mlp(X):
  X = X.reshape(X.shape[0], -1)
  H = torch.relu(X @ W1 + b1)
  return H @ W2 + b2

* Run the from-scratch MLP on a tiny batch.

In [ ]:
X = torch.randn(2, 1, 2, 2)
logits = mlp(X)
# X goes through reshape first to become (2, 4)
# X @ W1 + b1 = (2, 4) @ (4, 5) + (5, ) = (2, 5) or H (hidden layer); we add this hidden layer so that ReLU can be applied (adding irrational convexities to the model)
# H @ W2 + b2 = (2, 5) @ (5, 3) + (3, ) = (2, 3)

logits.shape

torch.Size([2, 3])

## 4. Step-by-step breakdown

*   `W1` and `b1` belong to the hidden layer.

*   `W2` and `b2` belong to the output layer.

*   The input is flattened into feature vectors.

*   `torch.relu` creates hidden activations.

*   The final line returns class logits.

## 5. Connection to ML systems

*   This is the same structure as a 2-layer neural classifier, just without PyTorch module objects.

## 6. Common confusion points

- Hidden parameters and output parameters have different shapes.
- The final output is logits, not probabilities.
- ReLU is applied after the hidden linear transformation.
- The model has no learning until gradients update parameters.

# 5.2.2 Concise Implementation

## 1. Intuition

* The concise implementation uses `torch.nn.Sequential`, `Flatten`, `Linear`, and `ReLU` modules.

* A module is a PyTorch object that stores parameters or computation behavior.

## 2. Why this exists

*   Concise modules reduce boilerplate and automatically register parameters for optimizers.

## 3. Examples

*   Define the same MLP with PyTorch modules.

In [ ]:
net = torch.nn.Sequential(
    torch.nn.Flatten(),         # Same as torch.reshape()
    torch.nn.Linear(4, 5),      # Hidden layer application to prepare for ReLU
    torch.nn.ReLU(),            # Apply ReLU to add irregular convexity
    torch.nn.Linear(5, 3),      # Final output layer
)
net(torch.randn(2, 1, 2, 2)).shape

torch.Size([2, 3])

*   Inspect parameter shapes.

In [ ]:
[p.shape for p in net.parameters()]

# W1 = (5, 4) given 4 input features and 5 output features in the first (hidden) Linear layer. We use W1.T or (4, 5) for X @ W1
# b1 = (5, )
# W2 = (3, 5) given 4 input features and 5 output features in the final Linear layer. We use W2.T or (5, 3) for X @ W2
# b2 = (3, )

[torch.Size([5, 4]), torch.Size([5]), torch.Size([3, 5]), torch.Size([3])]

## 4. Step-by-step breakdown


*   `Flatten` converts each image into a vector.

*   The first `Linear` module is the hidden layer.

*   `ReLU` applies the activation.

*   The second `Linear` module returns logits.

*   `net.parameters()` exposes trainable tensors.

## 5. Connection to ML systems

*   This is the standard way to define small MLPs in PyTorch.

## 6. Common confusion points

- `Sequential` hides no training logic; it only defines forward order.
- `ReLU` has no trainable parameters.
- `Linear` has weights and bias.
- Parameter shapes can be inspected before training.

# 5.2.3 Summary

## 1. Intuition

* From-scratch and concise MLPs compute the same kind of function.

* The concise version packages parameters and operations into modules.

## 2. Why this exists

* Mapping manual tensors to PyTorch modules keeps framework code understandable.

## 3. Examples

*   Map manual pieces to concise modules.

In [ ]:
mapping = {
    "W1, b1": "first (hidden) Linear layer",
    "ReLU call": "ReLU module",
    "W2, b2": "second (final output) Linear layer",
}
mapping

## 4. Step-by-step breakdown

* The dictionary names equivalent concepts.

* Manual parameters become `Linear` module parameters.

* Manual activation calls become activation modules.

## 5. Connection to ML systems

* Later architectures use the same mapping pattern with different layer types.

## 6. Common confusion points

- Concise code should be traceable to manual computation.
- Modules can store parameters or just computation.
- Output logits still need a task-appropriate loss (e.g. `MSELoss` for regression or `softmax` for classification problems).
- Hidden size is a hyperparameter.

  | | Hyperparameter | Parameter |
  |---|---|---|
  | Chosen by | Human/model designer | Training algorithm |
  | Updated by backprop? | No | Yes |
  | Examples | Hidden size, number of layers, learning rate | Weights, biases |
  | Exists before training? | Yes | Initialized before training, then updated |

# 5.2.4 Exercises

## 1. Intuition

* These exercises practice building and inspecting MLPs.

## 2. Why this exists

* Small architecture changes should be easy to reason about before training.

## 3. Examples

* Exercise 1: create a deeper MLP with two hidden layers.

In [15]:
deep = torch.nn.Sequential(
    torch.nn.Linear(3, 4),
    torch.nn.ReLU(),
    torch.nn.Linear(4, 4),
    torch.nn.ReLU(),
    torch.nn.Linear(4, 2),
)
print(deep(torch.randn(5, 3)).shape)
# X @ W1.T + b1 = (5, 3) @ (3, 4) + (4, ) = (5, 4), then ReLU to create irregular convexities
# X @ W2.T + b2 = (5, 4) @ (4, 4) + (4, ) = (5, 4), then ReLU to create irregular convexities
# X @ W3.T + b3 = (5, 4) @ (4, 2) + (2, ) = (5, 2)

print([p.shape for p in deep.parameters()])
# W1 = (4, 3), we use W1.T or (3, 4) for X @ W1
# b1 = (4, )
# W2 = (4, 4), we use W2.T or (4, 4) for X @ W1
# b2 = (4, )
# W2 = (2, 4), we use W2.T or (4, 2) for X @ W1
# b2 = (2, )

torch.Size([5, 2])
[torch.Size([4, 3]), torch.Size([4]), torch.Size([4, 4]), torch.Size([4]), torch.Size([2, 4]), torch.Size([2])]


* Exercise 2: count trainable tensors.

In [16]:
len(list(deep.parameters())) # 6

6

## 4. Step-by-step breakdown

* Exercise 1 checks layer chaining.

* Exercise 2 counts weight and bias tensors.

* 3 linear layers create 6 parameter tensors: 3 weights and 3 biases.

## 5. Connection to ML systems

* Parameter inspection becomes important for debugging larger networks.

## 6. Common confusion points

- Every Linear layer usually has 2 parameter tensors.
- ReLU layers do not add parameters.
- Adjacent layer dimensions must match.
- More layers add expressiveness and training difficulty.